# Compute Monthly Climatology

This notebook processes agroclim indicator files

## Processing Steps:
1. For each month (1-12):
   - Find all files for that month
   - Process in chunks of N years
   - Compute statistics incrementally
   - Save to NetCDF file
   - Clear memory before next month

## Setup and Configuration

In [1]:
# Import required libraries
import sys
import os
import gc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import process_single_month_optimized
import process_all_months_optimized

# Check available memory
import psutil
memory = psutil.virtual_memory()
print(f"\nSystem Memory:")
print(f"  Total: {memory.total / (1024**3):.1f} GB")
print(f"  Available: {memory.available / (1024**3):.1f} GB")
print(f"  Used: {memory.percent:.1f}%")

Optimized functions loaded successfully!
Python version: 3.12.5 | packaged by conda-forge | (main, Aug  8 2024, 18:36:51) [GCC 12.4.0]

System Memory:
  Total: 187.0 GB
  Available: 138.6 GB
  Used: 25.9%


In [2]:
# Configuration - UPDATE THESE PATHS FOR YOUR SYSTEM

# For cluster use:
DATA_DIR = "./monthly_indicators"
OUTPUT_DIR = "./monthly_indicators/monthly_climatology"

# For local testing (update to your actual path):
# DATA_DIR = "./monthly_indicators"
# OUTPUT_DIR = "./monthly_climatology"

# Variables to process
VARIABLES = ['gdd', 'hsd', 'frost_days']

# File pattern
FILE_PATTERN = "agroclim_indicator-*.nc"

# MEMORY OPTIMIZATION PARAMETER
CHUNK_YEARS = 23  # Process N years at a time (adjust as needed)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Variables to process: {VARIABLES}")
print(f"File pattern: {FILE_PATTERN}")
print(f"Chunk size: {CHUNK_YEARS} years at a time")

Data directory: ./monthly_indicators
Output directory: ./monthly_indicators/monthly_climatology
Variables to process: ['gdd', 'hsd', 'frost_days']
File pattern: agroclim_indicator-*.nc
Chunk size: 23 years at a time


## Check Available Data

In [3]:
# Check if data directory exists and list sample files
import glob

if os.path.exists(DATA_DIR):
    all_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))
    print(f"Found {len(all_files)} total files in {DATA_DIR}")
    
    if all_files:
        # Show first and last few files
        print("\nFirst 5 files:")
        for f in all_files[:5]:
            # Get file size
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")
        
        if len(all_files) > 10:
            print(f"\n  ... {len(all_files) - 10} more files ...")
        
        print("\nLast 5 files:")
        for f in all_files[-5:]:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")
        
        # Calculate total size
        total_size = sum(os.path.getsize(f) for f in all_files)
        print(f"\nTotal data size: {total_size / (1024**3):.2f} GB")
        
        # Extract year range
        years = set()
        for f in all_files:
            basename = os.path.basename(f)
            try:
                year_month = basename.split('-')[1].split('.')[0]
                year = int(year_month[:4])
                years.add(year)
            except:
                continue
        
        if years:
            print(f"\nYear range: {min(years)} - {max(years)}")
            print(f"Total years: {len(years)}")
            
            # Estimate memory requirement per month
            avg_file_size = total_size / len(all_files)
            files_per_month = len(years)
            memory_per_month = avg_file_size * files_per_month * 3  # Factor of 3 for processing overhead
            memory_per_chunk = avg_file_size * CHUNK_YEARS * 3
            
            print(f"\nMemory estimates:")
            print(f"  Per month (all years): {memory_per_month / (1024**3):.2f} GB")
            print(f"  Per chunk ({CHUNK_YEARS} years): {memory_per_chunk / (1024**3):.2f} GB")
            
            if memory_per_chunk > memory.available:
                suggested_chunk = max(1, int(CHUNK_YEARS * memory.available / memory_per_chunk))
                print(f"\n⚠️  WARNING: Estimated memory per chunk exceeds available memory!")
                print(f"  Consider reducing CHUNK_YEARS to {suggested_chunk}")
else:
    print(f"WARNING: Data directory does not exist: {DATA_DIR}")
    print("Please update DATA_DIR to point to your actual data location.")

Found 276 total files in ./monthly_indicators

First 5 files:
  - agroclim_indicator-200101.nc (19.6 MB)
  - agroclim_indicator-200102.nc (22.4 MB)
  - agroclim_indicator-200103.nc (26.5 MB)
  - agroclim_indicator-200104.nc (44.7 MB)
  - agroclim_indicator-200105.nc (59.1 MB)

  ... 266 more files ...

Last 5 files:
  - agroclim_indicator-202308.nc (75.2 MB)
  - agroclim_indicator-202309.nc (65.2 MB)
  - agroclim_indicator-202310.nc (51.8 MB)
  - agroclim_indicator-202311.nc (31.3 MB)
  - agroclim_indicator-202312.nc (25.9 MB)

Total data size: 12.82 GB

Year range: 2001 - 2023
Total years: 23

Memory estimates:
  Per month (all years): 3.20 GB
  Per chunk (23 years): 3.20 GB


## Process Single Month (Test with Memory Monitoring)

In [4]:
# # Test with January, monitoring memory usage --- WORKS FINE
# test_month = 1  # January

# if os.path.exists(DATA_DIR):
#     print(f"Testing with month {test_month:02d} (January)...\n")
    
#     # Check memory before
#     memory_before = psutil.virtual_memory()
#     print(f"Memory before processing: {memory_before.percent:.1f}% used")
    
#     try:
#         # Process single month with optimized function
#         output_files = process_single_month_optimized(
#             data_dir=DATA_DIR,
#             month=test_month,
#             output_dir=OUTPUT_DIR,
#             variables=VARIABLES,
#             pattern=FILE_PATTERN,
#             chunk_years=CHUNK_YEARS
#         )
        
#         if output_files:
#             print("\nOutput files created:")
#             for var, filepath in output_files.items():
#                 print(f"  {var}: {os.path.basename(filepath)}")
        
#         # Check memory after
#         memory_after = psutil.virtual_memory()
#         print(f"\nMemory after processing: {memory_after.percent:.1f}% used")
#         print(f"Memory increase: {memory_after.percent - memory_before.percent:.1f}%")
        
#         # Force garbage collection
#         gc.collect()
#         memory_gc = psutil.virtual_memory()
#         print(f"Memory after garbage collection: {memory_gc.percent:.1f}% used")
        
#     except MemoryError as e:
#         print(f"\n❌ Memory Error: {e}")
#         print(f"\nTry reducing CHUNK_YEARS to a smaller value (current: {CHUNK_YEARS})")
#         print("You can modify CHUNK_YEARS in the configuration cell above.")
#     except Exception as e:
#         print(f"Error processing month {test_month}: {e}")
# else:
#     print("Skipping test - data directory not found")

## Process All Months

In [ ]:
# Process all 12 months with memory optimization
if os.path.exists(DATA_DIR):
    print("Processing all months...\n")
    print("="*60)
    
    # Monitor initial memory
    initial_memory = psutil.virtual_memory()
    print(f"Starting memory usage: {initial_memory.percent:.1f}%\n")
    
    try:
        results = process_all_months_optimized(
            data_dir=DATA_DIR,
            output_dir=OUTPUT_DIR,
            variables=VARIABLES,
            months=None,  # None means process all 12 months
            pattern=FILE_PATTERN,
            chunk_years=CHUNK_YEARS
        )
        
        # Summary of results
        print("\n" + "="*60)
        print("PROCESSING SUMMARY")
        print("="*60)
        
        month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        
        for month in range(1, 13):
            if month in results:
                n_files = len(results[month])
                status = "✓" if n_files > 0 else "✗"
                print(f"  {month:02d} {month_names[month-1]}: {status} ({n_files} files)")
        
        # Final memory check
        final_memory = psutil.virtual_memory()
        print(f"\nFinal memory usage: {final_memory.percent:.1f}%")
        print(f"Total memory increase: {final_memory.percent - initial_memory.percent:.1f}%")
        
    except MemoryError as e:
        print(f"\n❌ Memory Error during processing: {e}")
        print(f"\nSuggestions:")
        print(f"1. Reduce CHUNK_YEARS to {max(1, CHUNK_YEARS - 1)}")
        print(f"2. Process months individually instead of all at once")
        print(f"3. Close other applications to free up memory")
        
else:
    print("Cannot process - data directory not found")
    print(f"Please ensure your data is in: {DATA_DIR}")

Processing all months with memory optimization...

Starting memory usage: 17.2%

Processing 12 months...
Input directory: ./monthly_indicators
Output directory: ./monthly_indicators/monthly_climatology
Chunk size: 23 years

Processing January (month 01)...
  Finding files for month 01...
  Found 23 files
  Year range: 2001-2023
  Computing climatological statistics (processing 23 years at a time)...
    Processing chunk 1/1 (23 files)
      gdd data shape: (1, 6500, 11700), dtype: float32
      gdd valid pixels in chunk: 29085674 / 76050000 (38.2%)
      hsd data shape: (1, 6500, 11700), dtype: float32
      hsd valid pixels in chunk: 29085674 / 76050000 (38.2%)
      frost_days data shape: (1, 6500, 11700), dtype: float32
      frost_days valid pixels in chunk: 29085674 / 76050000 (38.2%)
      gdd: mean=36.16, std=4.93, coverage=38.2%
      hsd: mean=0.75, std=0.24, coverage=38.2%
      frost_days: mean=23.23, std=0.96, coverage=38.2%
  Saving results...
      Saved: ./monthly_indica

## Verify Output Files

In [3]:
# Verify all output files were created
import glob
import xarray as xr
import numpy as np

print("Verifying output files...\n")

if os.path.exists(OUTPUT_DIR):
    output_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.nc")))
    
    if output_files:
        print(f"Found {len(output_files)} files in {OUTPUT_DIR}\n")
        
        # Organize by variable
        files_by_var = {}
        for var in VARIABLES:
            files_by_var[var] = [f for f in output_files if var in os.path.basename(f)]
        
        # Check completeness
        for var in VARIABLES:
            n_files = len(files_by_var[var])
            print(f"{var}:")
            print(f"  Expected: 12 monthly files")
            print(f"  Found: {n_files} files")
            
            if n_files == 12:
                print(f"  Status: ✓ Complete")
            else:
                print(f"  Status: ⚠ Incomplete")
                
                # Find missing months
                found_months = []
                for f in files_by_var[var]:
                    # Extract month from filename
                    basename = os.path.basename(f)
                    parts = basename.split('_')
                    for i, part in enumerate(parts):
                        if 'climatology' in part and i + 1 < len(parts):
                            try:
                                month = int(parts[i + 1].split('-')[0])
                                found_months.append(month)
                            except:
                                pass
                
                missing = set(range(1, 13)) - set(found_months)
                if missing:
                    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
                    missing_names = [month_names[m-1] for m in sorted(missing)]
                    print(f"  Missing: {', '.join(missing_names)}")
            print()
        
        # Sample one file to check content
        if files_by_var.get('gdd'):
            sample_file = files_by_var['gdd'][0]
            print(f"\nSample file: {os.path.basename(sample_file)}")
            
            with xr.open_dataset(sample_file) as ds:
                print(f"  Variables: {list(ds.data_vars)}")
                print(f"  Dimensions: {dict(ds.dims)}")
                
                # Check for NaN coverage
                for var in ds.data_vars:
                    if 'mean' in var:
                        data = ds[var].values
                        valid = np.sum(~np.isnan(data))
                        total = data.size
                        print(f"  {var} coverage: {100*valid/total:.1f}%")
    else:
        print(f"No output files found in {OUTPUT_DIR}")
else:
    print(f"Output directory does not exist: {OUTPUT_DIR}")

Verifying output files...

Found 36 files in ./monthly_indicators/monthly_climatology

gdd:
  Expected: 12 monthly files
  Found: 12 files
  Status: ✓ Complete

hsd:
  Expected: 12 monthly files
  Found: 12 files
  Status: ✓ Complete

frost_days:
  Expected: 12 monthly files
  Found: 12 files
  Status: ✓ Complete


Sample file: gdd_climatology_month01_2001-2023.nc
  Variables: ['gdd_mean', 'gdd_std', 'gdd_min', 'gdd_max', 'gdd_sum', 'gdd_year_count']
  Dimensions: {'lat': 6500, 'lon': 11700}
  gdd_mean coverage: 38.2%
